# Building a Simple AI Research Assistant

**Midterm laboratory — Phát triển các hệ thống thông minh (PTIT)**

This notebook is a small, complete **AI research assistant**. Given a research
topic, it asks an LLM (**DeepSeek**) to propose search queries, runs those
queries through a web search API (**Tavily**), and then asks DeepSeek to
summarize the findings and write a short research report — all inside one
Jupyter Notebook.

## Workflow

```text
                 Research Topic (you type it)
                            |
                            v
                 +---------------------+
                 |  DeepSeek LLM       |   <-- generates 3 search queries
                 +----------+----------+
                            |
                            v
                 +---------------------+
                 |  Tavily Web Search  |   <-- 3 queries x 3 results
                 +----------+----------+
                            |
                            v
                 +---------------------+
                 |  DeepSeek LLM       |   <-- summarizes the results
                 +----------+----------+
                            |
                            v
                 +---------------------+
                 |  DeepSeek LLM       |   <-- writes the final report
                 +----------+----------+
                            |
                            v
                    research_report.md
```

## Learning objectives

After finishing this lab you should be able to:

1. Call an external LLM API (DeepSeek) from Python.
2. Write **prompts** that make an LLM return a useful, structured answer.
3. Call a **web search API** (Tavily) and read its JSON response.
4. Combine search results with an LLM (**retrieval-augmented generation**).
5. Produce a Markdown research report inside a notebook.


## 01. Introduction — the four ideas behind this lab

**1. What is an LLM?**
A *Large Language Model* is a neural network trained on a huge amount of text
to predict the next token. It can summarize, translate, answer questions and
write code. In this lab the LLM provider is **DeepSeek** (not OpenAI/ChatGPT).
DeepSeek exposes an OpenAI-compatible API, so we can talk to it with a standard
chat-model client and only change the endpoint, the key and the model name.

**2. What is prompt engineering?**
The *prompt* is the text we send to the model. A good prompt states the role
("You are a research assistant"), the task, the input and the required output
format. Small wording changes produce very different answers, which is why we
write our prompts once, in named constants, and reuse them.

**3. What is tool calling?**
An LLM does **not** browse the Internet by itself. It only knows its training
data. To get fresh facts we give it a *tool* — here the Tavily search API:

```text
DeepSeek --> Search Tool --> Tavily --> Web Results --> DeepSeek
```

**4. What is Retrieval-Augmented Generation (RAG)?**
RAG is the simple pattern used by this notebook:

```text
Search external information  ->  put it in the prompt  ->  generate the answer
```

The model is instructed to use **only** the retrieved text, which reduces
hallucination and lets us cite real URLs. No vector database is needed here.


## 02. Install dependencies

The notebook needs only a few packages (see `requirements.txt`):

| Package | Why |
|---|---|
| `jupyter` | runs this notebook |
| `python-dotenv` | loads API keys from `.env` |
| `langchain-openai` | OpenAI-compatible chat client used for DeepSeek |
| `tavily-python` | Tavily web search client |

Run the cell below **once** if you have not installed them yet (it is commented
out so that "Run All" never reinstalls packages by accident).


In [1]:
# !pip install -r requirements.txt


## 03. Import libraries

Everything we need fits in seven imports. `ChatOpenAI` is used with a custom
`base_url`, which is how we point an OpenAI-compatible client at DeepSeek.


In [2]:
import json
import os
import re
import textwrap

from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_openai import ChatOpenAI
from tavily import TavilyClient

print("Libraries imported.")


/Users/papadracula141gmail.com/Desktop/pet-prj/LangGraph_Research_Assistant_Agent/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Libraries imported.


## 04. Configure API keys

API keys are **never** written in the notebook. They are read from environment
variables, which are loaded from a local `.env` file (copy `.env.example`).

```env
DEEPSEEK_API_KEY=...
TAVILY_API_KEY=...
DEEPSEEK_MODEL=deepseek-flash
```

The model name and the endpoint are configurable, so the notebook does not
hardcode a single model version. If a key is missing, the notebook prints a
clear message and switches on **DEMO MODE**: it then uses
`data/sample_results.json` and pre-written texts, so the whole notebook can
still be executed from top to bottom and the workflow is visible.


In [3]:
# Load variables from the local .env file (never hardcode secrets in a notebook).
load_dotenv()

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

# The model name is configurable; the current official DeepSeek models are
# "deepseek-flash" and "deepseek-v4-pro" (older tutorials use "deepseek-chat").
DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-flash")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")

# Keep API usage small: 3 queries x 3 results = at most 9 results.
MAX_RESULTS_PER_QUERY = 3

# DEMO MODE lets the notebook run end-to-end even without API keys.
DEMO_MODE = not (DEEPSEEK_API_KEY and TAVILY_API_KEY)

if not DEEPSEEK_API_KEY:
    print("DEEPSEEK_API_KEY is not configured.")
    print("Please configure your DeepSeek API key before running the notebook.")
if not TAVILY_API_KEY:
    print("TAVILY_API_KEY is not configured.")
    print("Please configure your Tavily API key before running the notebook.")

if DEMO_MODE:
    print("\nDEMO MODE is ON: sample data and pre-written texts will be used,")
    print("so every cell below still runs from top to bottom.")
else:
    print(f"\nConfiguration OK - DeepSeek model: {DEEPSEEK_MODEL}")



Configuration OK - DeepSeek model: deepseek-flash


## 05. Initialize the DeepSeek LLM

We create a **single** chat model for the whole notebook. DeepSeek is reached
through its OpenAI-compatible endpoint, so the only differences from an OpenAI
call are the `base_url`, the API key and the model name.

`temperature=0` keeps the answers factual and repeatable, which is what we want
for a research assistant.


In [4]:
if DEMO_MODE:
    llm = None
    print("DEMO MODE: no DeepSeek client is created (no API key available).")
else:
    llm = ChatOpenAI(
        model=DEEPSEEK_MODEL,
        api_key=DEEPSEEK_API_KEY,
        base_url=DEEPSEEK_BASE_URL,
        temperature=0,
    )
    print(f"DeepSeek LLM ready: {DEEPSEEK_MODEL} via {DEEPSEEK_BASE_URL}")


DeepSeek LLM ready: deepseek-flash via https://api.deepseek.com


## 06. Define the research topic

This is the only thing you have to edit to run the lab on a different subject.


In [5]:
topic = "Ứng dụng của trí tuệ nhân tạo trong giáo dục"

print("Research topic:", topic)


Research topic: Ứng dụng của trí tuệ nhân tạo trong giáo dục


## 07. Generate search queries with DeepSeek

The prompt asks the model for **three** short queries and for a numbered list,
so the answer is trivial to parse. `ask_deepseek()` is the single helper used
for every LLM call in this notebook; in DEMO MODE it returns a pre-written
answer instead of calling the API.

**Prompt used**

```text
You are a research assistant.
Given the following research topic:
{topic}
Generate {n} concise web search queries that will help research this topic.
Return only the search queries as a numbered list.
```

**Expected output (DEMO MODE)**

```text
1. AI applications in healthcare
2. AI medical diagnosis and imaging
3. AI in healthcare challenges and risks
```


In [6]:
QUERY_PROMPT = """You are a research assistant.

Given the following research topic:

{topic}

Generate {n} concise web search queries that will help research this topic.

Return only the search queries as a numbered list."""

# Used in DEMO MODE so the workflow can be demonstrated without an API key.
DEMO_QUERIES = """1. AI applications in healthcare
2. AI medical diagnosis and imaging
3. AI in healthcare challenges and risks"""


def ask_deepseek(prompt, demo_answer=""):
    """Send one prompt to DeepSeek and return the answer as plain text."""
    if DEMO_MODE:
        return demo_answer
    response = llm.invoke(prompt)
    return response.content.strip()


def generate_queries(topic, n=3):
    """Ask DeepSeek for n short web search queries about the topic."""
    prompt = QUERY_PROMPT.format(topic=topic, n=n)
    text = ask_deepseek(prompt, DEMO_QUERIES)

    queries = []
    for line in text.splitlines():
        # Remove list markers such as "1.", "1)", "-", "*".
        line = re.sub(r"^\s*(?:\d+[\.\)]|[-*])\s*", "", line).strip()
        if line:
            queries.append(line)
    return queries[:n]


queries = generate_queries(topic, n=3)

print(f"DeepSeek generated {len(queries)} search queries:")
for i, query in enumerate(queries, start=1):
    print(f"{i}. {query}")


DeepSeek generated 3 search queries:
1. ứng dụng trí tuệ nhân tạo trong giáo dục
2. AI trong dạy và học hiện nay
3. lợi ích và thách thức của AI trong giáo dục


## 08. Search the web with Tavily

Tavily returns JSON. For every query we keep only the three fields we need —
`title`, `url`, `content` — and we ignore the rest. Keeping the data structure
this small is what makes the rest of the notebook easy to read.

Basic error handling only: a failing query prints an error and returns an empty
list, so the notebook never crashes. An empty result prints the message
`No search results were found for this query.`


In [ ]:
# In DEMO MODE the search results are read from data/sample_results.json.
DEMO_RESULTS = {}
if DEMO_MODE:
    with open("data/sample_results.json", encoding="utf-8") as file:
        DEMO_RESULTS = json.load(file)


def search_web(query, max_results=MAX_RESULTS_PER_QUERY):
    """Search the web with Tavily and return [{title, url, content}, ...]."""
    if DEMO_MODE:
        results = DEMO_RESULTS.get(query, DEMO_RESULTS.get("_default", []))
        return results[:max_results]

    try:
        client = TavilyClient(api_key=TAVILY_API_KEY)
        response = client.search(query=query, max_results=max_results)
    except Exception as error:  # basic error handling, no retries
        print(f"Search failed for {query!r}: {error}")
        return []

    return [
        {
            "title": item.get("title", "Untitled"),
            "url": item.get("url", ""),
            "content": item.get("content", ""),
        }
        for item in response.get("results", [])
    ]


search_results = []
for query in queries:
    print(f"Searching: {query}")
    results = search_web(query, MAX_RESULTS_PER_QUERY)
    if not results:
        print("  No search results were found for this query.")
        continue
    for result in results:
        result["query"] = query
    search_results.extend(results)
    print(f"  -> {len(results)} result(s)")

print(f"\nTotal results collected: {len(search_results)}")


Searching: ứng dụng trí tuệ nhân tạo trong giáo dục
  -> 3 result(s)
Searching: AI trong dạy và học hiện nay
  -> 3 result(s)
Searching: lợi ích và thách thức của AI trong giáo dục


## 09. Display the search results

Each result is shown with its **title**, **URL** and a short **summary**, so a
human can quickly check that the retrieval step worked before spending LLM
tokens on it.


In [8]:
def show_search_results(results):
    """Render the collected search results as readable Markdown."""
    if not results:
        display(Markdown("_No search results to display._"))
        return

    blocks = []
    for i, item in enumerate(results, start=1):
        snippet = textwrap.shorten(item["content"], width=300, placeholder=" ...")
        blocks.append(
            f"**{i}. {item['title']}**\n\n"
            f"- URL: <{item['url']}>\n"
            f"- Query: `{item.get('query', '-')}`\n"
            f"- Summary: {snippet}"
        )
    display(Markdown("\n\n".join(blocks)))


show_search_results(search_results)


**1. Artificial Intelligence in Healthcare - World Health Organization**

- URL: <https://www.who.int/news-room/fact-sheets/detail/artificial-intelligence-in-health>
- Query: `AI applications in healthcare`
- Summary: Artificial intelligence (AI) is being adopted across health systems for diagnosis, treatment planning, drug discovery and health administration. WHO notes that AI can improve access to care in underserved regions, for example through mobile screening tools, but warns that benefits depend on ...

**2. AI in Health Care: Applications and Policy Considerations**

- URL: <https://www.mckinsey.com/industries/healthcare/our-insights/ai-in-health-care>
- Query: `AI applications in healthcare`
- Summary: Hospitals and life-science companies use AI mainly for administrative automation, clinical documentation, patient triage and imaging analysis. Reported productivity gains range from 10% to 30% in document-heavy workflows. Adoption barriers include fragmented data systems, unclear

## 10. Summarize the search results with DeepSeek

This is the "augmented generation" step: the retrieved text is inserted into
the prompt. The prompt forbids inventing facts and asks the model to mention
source URLs, which is the cheapest way to reduce hallucination.

**Prompt used**

```text
You are a research assistant.
Based only on the following search results, produce a concise and factual summary.
Do not invent information that is not supported by the provided sources.
Organize the information clearly, identify the most important findings,
and mention the source URLs when appropriate.
Research topic: {topic}
Search results: {results}
```


In [9]:
SUMMARY_PROMPT = """You are a research assistant.

Based only on the following search results, produce a concise and factual summary.

Do not invent information that is not supported by the provided sources.
Organize the information clearly, identify the most important findings,
and mention the source URLs when appropriate.

Research topic:
{topic}

Search results:
{results}"""

DEMO_SUMMARY = (
    "Artificial intelligence is used in healthcare mainly for medical imaging, "
    "diagnosis, drug discovery and administrative automation. Reported benefits "
    "include earlier detection, faster reporting and reduced documentation "
    "workload. The main challenges are sensitive-data privacy, algorithmic bias "
    "against under-represented populations, unclear accountability, and the need "
    "for clinician oversight before any automated recommendation is acted on. "
    "(Sample summary produced in DEMO MODE from data/sample_results.json.)"
)


def format_results(results):
    """Turn the result list into plain text that the LLM can read."""
    blocks = []
    for i, item in enumerate(results, start=1):
        content = textwrap.shorten(item["content"], width=1200, placeholder=" ...")
        blocks.append(f"[{i}] {item['title']}\nURL: {item['url']}\n{content}")
    return "\n\n".join(blocks)


def summarize_results(topic, results):
    """Ask DeepSeek for a factual summary of the search results."""
    if not results:
        return "No search results were available to summarize."
    prompt = SUMMARY_PROMPT.format(topic=topic, results=format_results(results))
    return ask_deepseek(prompt, DEMO_SUMMARY)


summary = summarize_results(topic, search_results)
print(summary)


Artificial intelligence is used in healthcare mainly for medical imaging, diagnosis, drug discovery and administrative automation. Reported benefits include earlier detection, faster reporting and reduced documentation workload. The main challenges are sensitive-data privacy, algorithmic bias against under-represented populations, unclear accountability, and the need for clinician oversight before any automated recommendation is acted on. (Sample summary produced in DEMO MODE from data/sample_results.json.)


## 11. Generate the final research report with DeepSeek

The summary is turned into a short Markdown report with a fixed structure. A
fixed structure is a simple but very effective prompt-engineering trick: the
model always returns the same sections, which makes the output predictable.


In [10]:
REPORT_PROMPT = """You are an AI research assistant.

Create a concise research report based only on the information provided below.

Topic:
{topic}

Research summary:
{summary}

Structure the report as:
1. Introduction
2. Main Findings
3. Key Applications / Examples
4. Challenges and Limitations
5. Conclusion

Use Markdown headings and do not invent unsupported facts."""

DEMO_REPORT = """# Research Report: Applications of Artificial Intelligence in Healthcare

## 1. Introduction
Artificial intelligence (AI) is increasingly used across healthcare, from
hospital administration to clinical decision support. This report summarizes
sample search results about where the technology is applied today and which
problems remain unsolved. It was generated in DEMO MODE, without calling the
DeepSeek API.

## 2. Main Findings
- Medical imaging is the most mature area: AI software is used for lesion
  detection, image enhancement and triage prioritization.
- Reported benefits include earlier detection, shorter reporting times and a
  10-30% reduction of effort in document-heavy administrative workflows.
- Clinical evidence is strong for narrow, well-defined tasks but generalization
  to new hospitals and populations is still limited.
- Data privacy, algorithmic bias and unclear accountability are the most
  frequently cited risks.

## 3. Key Applications / Examples
- Diabetic retinopathy and mammography screening.
- Radiology triage and pathology support.
- Clinical documentation and coding automation.
- Patient triage and remote monitoring.

## 4. Challenges and Limitations
- Training data is often small, unbalanced or unrepresentative.
- Model performance can degrade across demographic groups.
- Regulatory approval and reimbursement pathways remain unclear.
- Clinician oversight is required; AI assists rather than replaces experts.

## 5. Conclusion
AI already delivers measurable value in imaging and administrative workflows,
but safe adoption depends on representative data, external validation,
transparency and continuous monitoring after deployment. (Sample report
produced in DEMO MODE from data/sample_results.json.)"""


def generate_report(topic, summary):
    """Ask DeepSeek to write the final Markdown research report."""
    prompt = REPORT_PROMPT.format(topic=topic, summary=summary)
    return ask_deepseek(prompt, DEMO_REPORT)


report = generate_report(topic, summary)

print(f"Report generated ({len(report)} characters).")


Report generated (1751 characters).


## 12. Display the final report and save it

The report is rendered directly in the notebook with `IPython.display.Markdown`.
A `Sources` section is then appended **programmatically** from the real search
results, so the URLs are always correct and never hallucinated.

The same Markdown text is written to `research_report.md`.


In [11]:
def build_source_list(results):
    """Build a deduplicated Markdown source list from the search results."""
    seen = set()
    lines_out = []
    for item in results:
        if not item["url"] or item["url"] in seen:
            continue
        seen.add(item["url"])
        lines_out.append(f"- [{item['title']}]({item['url']})")
    return "\n".join(lines_out)


sources = build_source_list(search_results)
final_report = report.rstrip() + "\n\n## Sources\n\n" + (sources or "_No sources available._")

display(Markdown(final_report))

with open("research_report.md", "w", encoding="utf-8") as file:
    file.write(final_report)

print("\nThe final report was saved to research_report.md")


# Research Report: Applications of Artificial Intelligence in Healthcare

## 1. Introduction
Artificial intelligence (AI) is increasingly used across healthcare, from
hospital administration to clinical decision support. This report summarizes
sample search results about where the technology is applied today and which
problems remain unsolved. It was generated in DEMO MODE, without calling the
DeepSeek API.

## 2. Main Findings
- Medical imaging is the most mature area: AI software is used for lesion
  detection, image enhancement and triage prioritization.
- Reported benefits include earlier detection, shorter reporting times and a
  10-30% reduction of effort in document-heavy administrative workflows.
- Clinical evidence is strong for narrow, well-defined tasks but generalization
  to new hospitals and populations is still limited.
- Data privacy, algorithmic bias and unclear accountability are the most
  frequently cited risks.

## 3. Key Applications / Examples
- Diabetic retinopa

## 13. Conclusion

You have built a complete, working intelligent system in one notebook:

```text
Research topic
   -> DeepSeek generates search queries
   -> Tavily retrieves web results
   -> DeepSeek summarizes the results
   -> DeepSeek writes the final report
   -> research_report.md
```

**What to remember**

| Concept | Where it happens |
|---|---|
| LLM API call | cell 07, 10, 11 |
| Prompt engineering | `QUERY_PROMPT`, `SUMMARY_PROMPT`, `REPORT_PROMPT` |
| Tool / web search | `search_web()` (Tavily) |
| Retrieval-augmented generation | cell 10 (results inserted into the prompt) |
| Error handling | missing keys, failed search, empty result |

**Exercises**

1. Change `topic` to a subject of your choice (for example "AI in education")
   and run the notebook again.
2. Change `n` in `generate_queries(topic, n=3)` to 5 and observe the result.
3. Edit `REPORT_PROMPT` to add a "Future Work" section.
4. Print `len(search_results)` and compare it with the theoretical maximum
   (queries x `MAX_RESULTS_PER_QUERY`).

**Optional extensions (choose only one)**

- **A. LangGraph** — wrap the four steps in a `START -> ... -> END` graph.
- **B. More queries** — combine results from 5 queries instead of 3.
- **C. Export** — save the report as PDF, or as Markdown with a timestamp.
- **D. Citations** — cite each source inline as `[1]`, `[2]` in the report.

**Troubleshooting**

| Problem | Fix |
|---|---|
| `DEEPSEEK_API_KEY is not configured.` | copy `.env.example` to `.env` and add your key |
| `TAVILY_API_KEY is not configured.` | same, add your Tavily key |
| `ModuleNotFoundError` | run `pip install -r requirements.txt` |
| DEMO MODE is ON unexpectedly | an API key is missing; check your `.env` file |
| `Search failed for ...` | the network or the Tavily key is unavailable |
